<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/primeira_analise/light_gbm/Experimento%20(4_classes_com_pesos_manuais).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparação do ambiente

In [1]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


## Definição de X e y

In [3]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [4]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 5)
Teste: (28280, 5)


In [5]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

conama_status
Excelente    82775
Boa          21678
Atenção       7560
Crítica       1106
Name: count, dtype: int64
conama_status
Excelente    0.731752
Boa          0.191639
Atenção      0.066832
Crítica      0.009777
Name: proportion, dtype: float64


In [6]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
# teste 1
class_weights = {
    "Excelente": 1,
    "Boa": 2,
    "Atenção": 4,
    "Crítica": 8
}


In [7]:
#teste 2
class_weights = {
    "Excelente": 1,
    "Boa": 2,
    "Atenção": 2,
    "Crítica": 8
}


In [8]:

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                class_weight=class_weights,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [9]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(class_weight={'Atenção': 2, 'Boa': 2,
                                              'Crítica': 8, 'Excelente': 1},
                                n_jobs=-1, random_state=42, verbose=-1))])

## Avaliação das Métricas de Treino


In [ ]:
# MÉTRICAS DE TREINO teste 1
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7590413635198331
Train Precision:
0.7932096999371406
Train Recall:
0.7590413635198331
Train F1:
0.764401061186796

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.31      0.72      0.43      7560
         Boa       0.61      0.37      0.46     21678
     Crítica       0.30      0.49      0.37      1106
   Excelente       0.89      0.87      0.88     82775

    accuracy                           0.76    113119
   macro avg       0.53      0.61      0.54    113119
weighted avg       0.79      0.76      0.76    113119

Train Confusion Matrix:
[[ 5466   829   335   930]
 [ 5512  7932   506  7728]
 [  432    85   540    49]
 [ 6275  4159   417 71924]]


In [10]:
# MÉTRICAS DE TREINO teste 2
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7724431793067478
Train Precision:
0.785876804600016
Train Recall:
0.7724431793067478
Train F1:
0.7733422120263322

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.59      0.26      0.36      7560
         Boa       0.49      0.57      0.53     21678
     Crítica       0.22      0.64      0.33      1106
   Excelente       0.89      0.87      0.88     82775

    accuracy                           0.77    113119
   macro avg       0.55      0.59      0.52    113119
weighted avg       0.79      0.77      0.77    113119

Train Confusion Matrix:
[[ 1967  3655   801  1137]
 [  690 12294   867  7827]
 [   29   315   711    51]
 [  664  8869   836 72406]]


In [ ]:
# MÉTRICAS DE TESTE teste 1
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7448373408769449

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.28      0.66      0.39      1890
         Boa       0.57      0.34      0.43      5419
     Crítica       0.16      0.26      0.19       277
   Excelente       0.89      0.87      0.88     20694

    accuracy                           0.74     28280
   macro avg       0.47      0.53      0.47     28280
weighted avg       0.78      0.74      0.75     28280


Confusion Matrix:
[[ 1240   262   116   272]
 [ 1424  1852   158  1985]
 [  155    41    71    10]
 [ 1582  1101   110 17901]]


In [11]:
# MÉTRICAS DE TESTE teste 2
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7578500707213579

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.50      0.23      0.31      1890
         Boa       0.46      0.53      0.50      5419
     Crítica       0.13      0.39      0.20       277
   Excelente       0.88      0.87      0.88     20694

    accuracy                           0.76     28280
   macro avg       0.50      0.50      0.47     28280
weighted avg       0.77      0.76      0.76     28280


Confusion Matrix:
[[  426   945   217   302]
 [  214  2890   276  2039]
 [   23   128   107    19]
 [  181  2287   217 18009]]


## Análise do balanceamento manual com quatro classificações

Com o objetivo de encontrar um equilíbrio melhor entre as classes do conjunto de dados, foi realizado um experimento utilizando pesos definidos manualmente para cada categoria:

```python
class_weights = {
    "Excelente": 1,
    "Boa": 2,
    "Atenção": 4,
    "Crítica": 8
}
```

A motivação desse teste surgiu após os resultados obtidos com o parâmetro `class_weight='balanced'`, que havia aumentado significativamente o recall da classe Crítica, porém ao custo de uma queda expressiva na precisão, indicando excesso de falsos positivos.

### Resultados obtidos

No conjunto de treino, o modelo apresentou:

* Accuracy: 75,9%
* Precision ponderada: 79,3%
* Recall ponderado: 75,9%
* F1-score ponderado: 76,4%

No conjunto de teste, os resultados foram:

* Accuracy: 74,5%
* Precision ponderada: 78%
* Recall ponderado: 74%
* F1-score ponderado: 75%

A diferença entre treino e teste permaneceu pequena, sugerindo que o modelo continuou apresentando boa capacidade de generalização e sem sinais relevantes de overfitting.

### Comportamento das classes

A classe Excelente continuou apresentando os melhores resultados, com precisão e recall próximos de 90%, comportamento esperado devido à sua elevada representatividade no conjunto de dados.

Por outro lado, as classes Atenção e Crítica continuaram sendo as mais desafiadoras.

A classe Crítica apresentou:

* Precisão: 16%
* Recall: 26%
* F1-score: 19%

Embora esses valores ainda estejam distantes do ideal, representam uma situação mais equilibrada quando comparados ao experimento utilizando `class_weight='balanced'`, no qual o modelo passou a identificar um número excessivo de amostras como Crítica.

### Análise da matriz de confusão

A matriz de confusão mostrou que o principal erro continua ocorrendo entre as classes Crítica e Atenção.

Dos 277 casos críticos presentes no conjunto de teste:

* 71 foram corretamente identificados como Crítica;
* 155 foram classificados como Atenção;
* apenas uma pequena parcela foi confundida com Excelente.

Esse comportamento é particularmente relevante porque indica que o modelo não está confundindo amostras críticas com situações claramente adequadas. O erro ocorre principalmente entre duas classes que representam níveis intermediários de qualidade ambiental.

Esse resultado sugere uma possível sobreposição entre as características das classes Atenção e Crítica dentro das variáveis disponíveis para treinamento.

### Interpretação dos resultados

O uso dos pesos manuais permitiu reduzir o comportamento excessivamente agressivo observado no balanceamento automático, produzindo um equilíbrio mais adequado entre precisão e recall.

Entretanto, os resultados indicam que o ajuste de pesos, isoladamente, não resolve completamente o problema. A principal dificuldade do modelo continua sendo diferenciar amostras classificadas como Atenção e Crítica.

Esse comportamento reforça a hipótese de que a limitação pode estar relacionada não apenas ao desbalanceamento das classes, mas também à sobreposição de características entre essas categorias ou à influência de fatores externos ainda não investigados, como a distribuição temporal das amostras ao longo dos anos.


Foram realizados diversos experimentos variando os pesos das classes no LightGBM. Embora os ajustes tenham alterado significativamente o equilíbrio entre precisão e recall da classe Crítica, todos os cenários apresentaram o mesmo padrão de erro: grande parte das amostras críticas continua sendo confundida com a classe Atenção. Isso sugere que o problema não está apenas na estratégia de balanceamento utilizada, mas possivelmente na própria estrutura dos dados ou na definição dos rótulos, indicando uma forte sobreposição entre as classes Atenção e Crítica.